# Textanalyse

## Kodierung

<div class="alert alert-block alert-info">

**Siehe auch:**

* [Unicode und Zeichenkodierungen](https://python-basics-tutorial.readthedocs.io/de/latest/types/strings/encodings.html)

</div>

### Problematische Steuerzeichen

Zunächst mag es sinnvoll erscheinen, beliebige Unicode-Zeichen in Zeichenketten zuzulassen. Dies ist jedoch selten ratsam. Der [RFC 9839: Unicode Character Repertoire Subsets](https://www.rfc-editor.org/info/rfc9839/) werden drei Klassen problematischer Codepunkte definiert: [veraltete Steuercodes](https://www.rfc-editor.org/info/rfc9839/#name-legacy-controls), [Nicht-Zeichen](https://www.rfc-editor.org/info/rfc9839/#name-noncharacters) und [Surrogatzeichen](https://www.rfc-editor.org/info/rfc9839/#name-surrogates). Sie können zu unübersichtlichen oder verwirrenden Datenanalysen führen oder Probleme bei der Verarbeitung bereiten und sollten daher gekennzeichnet, ersetzt oder entfernt werden.

### Textkodierung angeben

Gebt beim Einlesen von Text die richtige Textkodierung an:

In [1]:
text = "El Niño"

text.encode("utf-8")

b'El Ni\xc3\xb1o'

### Normalisieren des Textes

Die Python-Bibliothek [charset-normalizer](https://pypi.org/project/charset-normalizer/) kann euch beim Ermitteln der richtigen Kodierungen helfen. Ihr könnt die Bibliothek verwenden, z. B. mit:

```console
uv add charset-normalizer
```

In [2]:
!normalizer ../../../data/iot_example.json

{
    "path": "/Users/veit/cusy/trn/Python4DataScience-de/data/iot_example.json",
    "encoding": "ascii",
    "encoding_aliases": [
        "646",
        "ansi_x3.4_1968",
        "ansi_x3_4_1968",
        "ansi_x3.4_1986",
        "cp367",
        "csascii",
        "ibm367",
        "iso646_us",
        "iso_646.irv_1991",
        "iso_ir_6",
        "us",
        "us_ascii"
    ],
    "alternative_encodings": [],
    "language": "English",
    "alphabets": [
        "Basic Latin",
        "Control character"
    ],
    "has_sig_or_bom": false,
    "chaos": 0.0,
    "coherence": 0.0,
    "unicode_path": null,
    "is_preferred": true
}


oder

In [3]:
from charset_normalizer import from_path


iot_example = from_path("../../../data/iot_example.json")
print(str(iot_example.best()))

{
    "creation_metadata": {
        "local_time": "2026-09-15T14:31:16",
        "utc_time": "2026-09-15T12:31:16+00:00",
        "creator": "TDDA 2.2.17",
        "host": "fay.local",
        "user": "veit",
        "n_records": 146397,
        "n_selected": 146397
    },
    "fields": {
        "timestamp": {
            "type": "string",
            "min_length": 19,
            "max_length": 19,
            "max_nulls": 0,
            "no_duplicates": true
        },
        "username": {
            "type": "string",
            "min_length": 3,
            "max_length": 21,
            "max_nulls": 0
        },
        "temperature": {
            "type": "int",
            "min": 5,
            "max": 29,
            "sign": "positive",
            "max_nulls": 0
        },
        "heartrate": {
            "type": "int",
            "min": 60,
            "max": 89,
            "sign": "positive",
            "max_nulls": 0
        },
        "build": {
            "type": "s

Nicht alle Texte sind jedoch eindeutig kodiert:

> Assume all external input is the result of (a series of) bugs.

– [RFC 9225](https://www.rfc-editor.org/rfc/rfc9225.html)

Hier kann euch [ftfy](https://ftfy.readthedocs.io/en/latest/) unterstützen:

```console
uv add ftfy
```

In [4]:
import ftfy


ftfy.fix_text("l’humanitÃ©")

"l'humanité"

Üblicherweise wird auf die *Normalization Form Canonical Composition* `NFC` oder auf die *Normalization Form Compatibility Composition* `NFKC` normalisiert:

In [5]:
import unicodedata


nfkc = unicodedata.normalize("NFKC", text)

"; ".join(f"U+{ord(c):04X}: {unicodedata.name(c)}" for c in nfkc)

'U+0045: LATIN CAPITAL LETTER E; U+006C: LATIN SMALL LETTER L; U+0020: SPACE; U+004E: LATIN CAPITAL LETTER N; U+0069: LATIN SMALL LETTER I; U+00F1: LATIN SMALL LETTER N WITH TILDE; U+006F: LATIN SMALL LETTER O'

Bevor ihr Texte miteinander vergleicht, sollten beide dieselbe Normalisierungsform haben:

In [6]:
nfkd = unicodedata.normalize("NFKD", text)

nfkc == nfkd

False

Auch die Länge der Texte ist abhängig von der Normalisierungsform:

In [7]:
len(nfkc) == len(nfkd)

False

<div class="alert alert-block alert-info">

**Siehe auch:**

* [Normalisierung (Unicode)](https://de.wikipedia.org/wiki/Normalisierung_(Unicode))

</div>

## String-Matching

### `difflib`

[String-Matching-Algorithmen](https://de.wikipedia.org/wiki/String-Matching-Algorithmus) werden zum Finden von Textsegmenten in einer Zeichenkette anhand eines Suchmusters verwendet. Suchmuster können in Python z. B. mit dem [re](https://python-basics-tutorial.readthedocs.io/en/latest/types/strings/built-in-modules/re.html)-Modul angegeben werden.

Ihr könnt jedoch auch mit der [get_close_matches](https://docs.python.org/3/library/difflib.html#difflib.get_close_matches)-Option der Python [difflib](https://docs.python.org/3/library/difflib.html)-Bibliothek Zeichenketten miteinander vergleichen.

In [8]:
import difflib


options = [
    "Berlin",
    "Berlin, Germany",
    "Berlin, Deutschland",
    "Berlin, DE",
    "Bundeshauptstadt",
    "Spreeathen",
]

difflib.get_close_matches("Brln", options)

['Berlin']

Wenn ihr festlegen wollt, wie groß die Übereinstimmung mindestens sein muss, damit ein Vorschlag angezeigt wird, könnt ihr den Standardwert des Arguments `cutoff` von `0.6` ändern:

In [9]:
difflib.get_close_matches("Brln", options, cutoff=0.5)

['Berlin', 'Berlin, DE']

<div class="alert alert-block alert-info">

**Bemerkung:**

In Python 3.14 erhielt `argparse` die neue Option [suggest_on_error](https://docs.python.org/3/library/argparse.html#suggest-on-error), die sich genau darauf stützt.

</div>

Mit dem `difflib` [SequenceMatcher](https://docs.python.org/3/library/difflib.html#difflib.SequenceMatcher) können wir uns auch die Ähnlichkeit von zwei Zeichenketten berechnen lassen:

In [10]:
from difflib import SequenceMatcher


m = SequenceMatcher(None, "Brln", options[0])

m.ratio()

0.8

Beide Zeichenfolgen `Brlin` und `Berlin` scheinen zu 80 % identisch. Die Standardmessung der *Zeichenfolgenähnlichkeit* ist zwar gut für einzelne Wörter und lange Zeichenfolgen, aber für kurze Zeichenketten mit zwei bis zehn Wörtern weniger gut geeignet. Der naive Ansatz reagiert viel zu empfindlich auf geringfügige Unterschiede in der Wortreihenfolge, fehlende oder zusätzliche Wörter und andere derartige Probleme:

In [11]:
m = SequenceMatcher(None, "Brln", options[1])

m.ratio()

0.42105263157894735

### `TheFuzz`

[TheFuzz](https://github.com/seatgeek/thefuzz) geht über die Möglichkeiten der `difflib` hinaus. Die verschiedenen verfügbaren Methoden und Unterschiede sind im Blogbeitrag [FuzzyWuzzy: Fuzzy String Matching in Python](https://chairnerd.seatgeek.com/fuzzywuzzy-fuzzy-string-matching-in-python/) beschrieben.

#### 1. Installation

Mit [uv](../../productive/envs/uv/index.rst) könnt ihr `TheFuzz` und die optionale [python-levenshtein](https://pypi.org/project/python-Levenshtein/)-Bibliothek in eurem Kernel bereitstellen:

```console
$ uv add thefuzz
```

#### 2. Import

In [12]:
from thefuzz import fuzz, process

#### 3. Beispiel

##### 3.1 String-Ähnlichkeit

Auch hier kommt die Berechnung der *Zeichenfolgenähnlichkeit* zunächst zum selben Ergebnis:

In [13]:
fuzz.ratio("Brln", options[1])

42

##### 3.2 Partielle String-Ähnlichkeit

Wir können hier jedoch auch eine Heuristik verwenden, die als _best partial_ bezeichnet wird.

In [14]:
fuzz.partial_ratio("Brln", options[1])

75

##### 3.3 Token-Sortierung

Bei der Token-Sortierung wird die betreffende Zeichenfolge mit einem Token versehen, die Token alphabetisch sortiert und anschließend wieder zu einer Zeichenfolge zusammengefügt, beispielsweise:

In [15]:
fuzz.token_set_ratio("Brln", options[1])

44

##### 3.4 Extrahieren aus einer Liste

In [16]:
process.extract("Brln", options, limit=1)

[('Berlin', 80)]

In [17]:
process.extract("Brln", options)

[('Berlin', 80),
 ('Berlin, Germany', 68),
 ('Berlin, Deutschland', 68),
 ('Berlin, DE', 68),
 ('Bundeshauptstadt', 51)]

#### 4. Weitere Informationen

In [18]:
process.extract?

Signature:
process.extract(
    query,
    choices,
    processor=<function full_process at 0x106926b60>,
    scorer=<function WRatio at 0x106927420>,
    limit=5,
)
Docstring:
Select the best match in a list or dictionary of choices.

Find best matches in a list or dictionary of choices, return a
list of tuples containing the match and its score. If a dictionary
is used, also returns the key for each match.

Arguments:
    query: An object representing the thing we want to find.
    choices: An iterable or dictionary-like object containing choices
        to be matched against the query. Dictionary arguments of
        {key: value} pairs will attempt to match the query against
        each value.
    processor: Optional function of the form f(a) -> b, where a is the query or
        individual choice and b is the choice to be used in matching.

        This can be used to match against, say, the first element of
        a list:

        lambda x: x[0]

        Defaults to thefuzz.util

<div class="alert alert-block alert-info">

**Siehe auch:**

Typische Algorithmen für String-Matching sind:

* [Levenshtein-Distanz](https://de.wikipedia.org/wiki/Levenshtein-Distanz)
* [Gestalt Pattern Matching](https://de.wikipedia.org/wiki/Gestalt_Pattern_Matching)

Auch für ähnlich klingende Wörter gibt es entsprechende Algorithmen:

* [ Soundex](https://de.wikipedia.org/wiki/Soundex)
* [Kölner Phonetik](https://de.wikipedia.org/wiki/Kölner_Phonetik)

</div>

#### 5. Bekannte Ports

FuzzyWuzzy wird auch in andere Sprachen portiert! Hier einige bekannte Ports:

* Java: [xpresso](https://github.com/WantedTechnologies/xpresso)
* Java: [xdrop fuzzywuzzy](https://github.com/xdrop/fuzzywuzzy)
* Rust: [fuzzyrusty](https://github.com/logannc/fuzzywuzzy-rs)
* JavaScript: [fuzzball.js](https://github.com/nol13/fuzzball.js)
* C++: [tmplt fuzzywuzzy](https://github.com/Tmplt/fuzzywuzzy)
* C#: [FuzzySharp](https://github.com/BoomTownRoi/BoomTown.FuzzySharp)
* Go: [go-fuzzywuzzy](https://github.com/paul-mannino/go-fuzzywuzzy)
* Pascal: [FuzzyWuzzy.pas](https://github.com/DavidMoraisFerreira/FuzzyWuzzy.pas)
* Kotlin: [FuzzyWuzzy-Kotlin](https://github.com/jens-muenker/fuzzywuzzy-kotlin)
* R: [fuzzywuzzyR](https://github.com/mlampros/fuzzywuzzyR)

### textacy

[textacy](https://github.com/chartbeat-labs/textacy) nutzt für Aufgaben wie [Tokenisierung](https://de.wikipedia.org/wiki/Tokenisierung), [Part-of-Speech-Tagging](https://de.wikipedia.org/wiki/Part-of-speech-Tagging), [Dependenzparsing](https://de.wikipedia.org/wiki/Dependenzparsing) die [spaCy](https://spacy.io/)-Bibliothek.

Ihr könnt über praktische Methoden und benutzerdefinierte Erweiterungen auf die Kernfunktionalität von spaCy zugreifen und erweitern, sodass ihr einfach ein oder mehrere Dokumente analysieren könnt:

1. Laden vorbereiteter Datensätze mit Textinhalten und Metadaten
2. Textinhalte bereinigen, normalisieren und im Rohtext untersuchen, bevor sie mit spaCy weiterverarbeitet werden
3. Extrahieren strukturierter Informationen aus verarbeiteten Dokumenten, darunter [N-Gramme](https://de.wikipedia.org/wiki/N-Gramm), [Entitäten](https://de.wikipedia.org/wiki/Entit%C3%A4t_(Informatik)), [Akronyme](https://de.wikipedia.org/wiki/Akronym), [Bestimmungsschlüssel](https://de.wikipedia.org/wiki/Bestimmungsschl%C3%BCssel) und [SVO-Tripel](https://de.wikipedia.org/wiki/Subjekt-Verb-Objekt).
4. Vergleichen von Zeichenfolgen und Sequenzen mithilfe verschiedener Ähnlichkeitsmetriken
5. Tokenisieren und vektorisieren von Dokumenten, wodurch anschließend Themenmodelle trainiert, interpretiert und visualisiert werden können  
6. Berechnen von Statistiken zur Lesbarkeit und lexikalischen Vielfalt von Texten, darunter den [Flesch-Kincaid-Lesbarkeitsindex](https://de.wikipedia.org/wiki/Lesbarkeitsindex#Flesch-Kincaid-Grade-Level), den [mehrsprachigen Flesch Reading Ease-Index](https://de.wikipedia.org/wiki/Lesbarkeitsindex#Flesch-Reading-Ease) und das [Type-Token-Verhältnis](https://de.wikipedia.org/wiki/Token_und_Type)